# Match labelled rows with LLM-extracted rows
1. Load data
2. Vectorize rows
3. Match rows 
4. Compute accuracy


In [10]:
import datetime as dt
import geopandas as gpd
from matplotlib import pyplot as plt
from src.data import *
from src.post_process_functions import *
from src.hazard_def import *
from src.impact_def import *
from src.accuracy import *

## Load data

In [11]:
#load data (model)
split_lowest = False
suffix = "_geo_split_lowest" if split_lowest else "_geo"
models = ["meta-llama_llama-4-scout-17b-16e-instruct", "llama-3.1-8b-instant", "llama-3.3-70b-versatile", "openai_gpt-oss-20b"]
#"llama-3.1-8b-instant"
# #"llama-3.3-70b-versatile"
# "meta-llama_llama-4-scout-17b-16e-instruct"
res_savename_ext = f"merged_subtypes_post_processed_labelled_reports_fixed_impact_desc3_{models[0]}_v271025{suffix}_v271025"
#f"merged_subtypes_post_processed_new_unit_std_labelled_reports_turnoff_subtype_val_{models[0]}_v141025{suffix}_v171025"
#f"merged_subtypes_post_processed_labelled_reports_fixed_impact_desc3_{model}_v271025{suffix}_v271025"#merged_subtypes_
#f"post_processed_flags_labelled_reports_{model}_v230925"
#"post_processed_llm_response_impact_labelled_reports_test_multiprompt_continue_v050925_21rep_meta-llama_llama-4-scout-17b-16e-instruct"
#"post_processed_labelled_reports_test_date_meta-llama_llama-4-scout-17b-16e-instruct_v180925"
#extracted_df_no_geo = pd.read_csv(DATA_OUT_PROC / (res_savename_ext+".csv"))

#load data (labelled)
res_savename_lab = f"merged_subtypes_post_processed_new_unit_std_labelled_reports_impacts_all_v111025{suffix}_v171025"#merged_subtypes_
#"post_processed_flags_labelled_reports_impacts_all_v240925"
# "post_processed_labelled_reports_impacts_all_v080925"
#labelled_df_no_geo = pd.read_csv(DATA_OUT_PROC / (res_savename_lab+".csv"))
#load geocoded data
extracted_df = gpd.read_file(DATA_OUT_PROC / (res_savename_ext+".gpkg"))#(res_savename_ext+suffix+suffix2+".gpkg")
#load data (labelled)
labelled_df = gpd.read_file(DATA_OUT_PROC / (res_savename_lab+".gpkg"))#(res_savename_lab+suffix+suffix2+".gpkg")

In [12]:
#reformat output
num_cols = ["impactValue","startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
list_cols_ext = ["country", "location", "hazards", "valueAnnotation", "locationAnnotation", "dateAnnotation", "hazardsAnnotation"]
list_cols_lab = ["country", "location", "hazards", "annotation"]
labelled_df = format_output(labelled_df, num_cols=num_cols, list_cols=list_cols_lab)
extracted_df = format_output(extracted_df, num_cols=num_cols, list_cols=list_cols_ext)

##for matching, need to replace NaNs in dates and units as np.nan cannot be compared
labelled_df[["startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]] = labelled_df[["startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]].fillna(-1)
extracted_df[["startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]] = extracted_df[["startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]].fillna(-1)
labelled_df["impactUnit"] = labelled_df["impactUnit"].fillna("null")
extracted_df["impactUnit"] = extracted_df["impactUnit"].fillna("null")

DEF_CRS_EPSG = "EPSG:4326"
labelled_df = labelled_df.set_crs(DEF_CRS_EPSG, allow_override=True)
extracted_df = extracted_df.set_crs(DEF_CRS_EPSG, allow_override=True)
combined_df = pd.concat([labelled_df, extracted_df])


## Match
1. Vectorize columns that need to be compared using cosine similarity
2. Compute cosine similarity for those columns for each possible extracted-labelled pair
3. Add absolute difference of impactValue between each possible extracted-labelled pair.
    Need to consider NaN from not NaN separately. Only try matching non-NaNs with non-NaNs 
    (and nans with nan?)
4. Compute Intersection-Over-Union of polygons for each possible pair
5. Match by maximizing similarity and -impactvalu_idff and -IoT. Allow for more than one match.  

In [13]:
# weights for matching
weight_set = dict()
weight_set["ws1"] = {#only impsubtype and unit
    'hazards' : 0,
    'iso3_code' : 0,
    'startYear' : 0,
    'startMonth' : 0,
    'startDay' : 0,
    'endYear' : 0,
    'endMonth' : 0,
    'endDay' : 0,
    'impactSubtype' : 1,
    'impactUnit' : 1,
    'geometry' : 0, #weight for geometry matching
    'impactValue' : 0
    }
weight_set["ws2"] = {#biased towards impsubtype and impunit
    'hazards' : 2,
    'iso3_code' : 2,
    'startYear' : 2,
    'startMonth' : 1,
    'startDay' : 1,
    'endYear' : 1,
    'endMonth' : 1,
    'endDay' : 1,
    'impactSubtype' : 20, #bigger than sum of all other weights (24)
    'impactUnit' : 20,
    'geometry' : 5, #weight for geometry matching
    'impactValue' : 5
    }

weight_set["ws3"] = {#flat
    'hazards' : 1,
    'iso3_code' : 1,
    'startYear' : 1,
    'startMonth' : 1,
    'startDay' : 1,
    'endYear' : 1,
    'endMonth' : 1,
    'endDay' : 1,
    'impactSubtype' : 1, #bigger than sum of all other weights (12)
    'impactUnit' : 1,
    'geometry' : 1, #weight for geometry matching
    'impactValue' : 1
    }
weight_set["ws4"] = {#only impsubtype
    'hazards' : 0,
    'iso3_code' : 0,
    'startYear' : 0,
    'startMonth' : 0,
    'startDay' : 0,
    'endYear' : 0,
    'endMonth' : 0,
    'endDay' : 0,
    'impactSubtype' : 1,
    'impactUnit' : 0,
    'geometry' : 0, #weight for geometry matching
    'impactValue' : 0
    }

weight_set["ws5"] = {#for quali data, subtype + geom > sum of other weights (12)
    'hazards' : 2,
    'iso3_code' : 2,
    'startYear' : 2,
    'startMonth' : 1,
    'startDay' : 1,
    'endYear' : 1,
    'endMonth' : 1,
    'endDay' : 1,
    'impactSubtype' : 10,
    'impactUnit' : 1,
    'geometry' : 5,
    'impactValue' : 0
    }
weight_set["ws6"] = {#for quanti data, 2 of impSub, impUn or impVal > sum of other weights (16)
    'hazards' : 2,
    'iso3_code' : 2,
    'startYear' : 2,
    'startMonth' : 1,
    'startDay' : 1,
    'endYear' : 1,
    'endMonth' : 1,
    'endDay' : 1,
    'impactSubtype' : 12,
    'impactUnit' : 12,
    'geometry' : 5,
    'impactValue' : 6
    }

In [14]:
#define target variables for (cosine) similarity calculation
UNIQUE_COUNTRIES_ISO = [country.alpha_3 for country in pycountry.countries]
UNIQUE_COUNTRY_NAMES = [country.name for country in pycountry.countries]

UNIQUE_DICT = {#mapping dictonary of unique values to generate vectors for cosine similarity
    'hazards' : list(hazard_main_types_emdat_desc.keys()),
    #'country' : UNIQUE_COUNTRY_NAMES,
    'iso3_code' : UNIQUE_COUNTRIES_ISO,
    'startYear' : np.arange(1980, 2026).tolist()+[-1],
    'startMonth' : np.arange(1, 13).tolist()+[-1],
    'startDay' : np.arange(1, 32).tolist()+[-1],
    'endYear' : np.arange(1980, 2026).tolist()+[-1],
    'endMonth' : np.arange(1, 13).tolist()+[-1],
    'endDay' : np.arange(1, 32).tolist()+[-1],
    'impactSubtype' : IMPACT_SUBTYPES_MERGED,
    'impactUnit' : combined_df.impactUnit.unique().tolist(),
}

SIMILARITY_VARS = list(UNIQUE_DICT.keys()) #all cols for which cosine similarity needs to be computed

In [15]:
#extracted_df.replace({"Access to Transport and Mobility" : "Mobility and Access to Transport",
#                      "Access to transport and Mobility" : "Mobility and Access to Transport",
#                      "Agriculture Infrastructure" : "Agricultural Infrastructure",
#                      "Other Economic and Livelihood Impacts" : "Other Economic Activity & Livelihood Production",
#                      "Informal settlements" : "Informal Settlements"}, inplace=True)
#labelled_df.replace({"Access to Transport and Mobility" : "Mobility and Access to Transport",
#                      "Access to transport and Mobility" : "Mobility and Access to Transport",
#                      "Agriculture Infrastructure" : "Agricultural Infrastructure",
#                      "Other Economic and Livelihood Impacts" : "Other Economic Activity & Livelihood Production",
#                      "Informal settlements" : "Informal Settlements"}, inplace=True)
#labelled_df = extracted_df


In [16]:
#extracted_df = extracted_df[extracted_df["appealCode"]=="MDRS2001"]

In [17]:
## Parameters
geo_match = True
value_match = "pre"#"pre", "post" minimize diff of impactValue simultaneously as cat columns (pre) or after matching of cat columns (post)
impactValue_error_clip = None
nan_policy = "strict" # "loose", "strict" #allow NaNs from one side to be matched with all rows from the other side

#select variable for matching
matching_cols = list() + SIMILARITY_VARS #cols used for matching

#choose weights
ws_key_qt = "ws6"
ws_key_ql = "ws5"
matching_cols_weights_qt = weight_set[ws_key_qt] # weights for matching quantitative
matching_cols_weights_ql = weight_set[ws_key_ql] # weights for matching qualitative

#updating matching columns and weights according to parameters
if geo_match:
    matching_cols.append("geometry")
if value_match == "pre":
    matching_cols.append("impactValue")
else:
    impactValue_error_clip = (0,1)

similarity_cols = [col for col in matching_cols if col in SIMILARITY_VARS]

#saving params
save_results = True
sim_name = f"{ws_key_qt}qt-{ws_key_ql}ql-geo-value{value_match}-{nan_policy}-nans"
filename_out = f"matched_data_{sim_name}_{res_savename_ext}"


In [ ]:
##Matching
match_idx = []

for appeal, ext_group in extracted_df.groupby("appealCode"):
    print("Processing appeal", appeal)
    ext_group = ext_group.reset_index(drop=False, names=["orig_index"]) #need to reset index to get indices for numpy arrays
    lab_group = labelled_df[labelled_df["appealCode"] == appeal].reset_index(drop=False, names=["orig_index"])

    if (lab_group.shape[0] == 0) or (ext_group.shape[0] == 0):
        continue

    #vectorize
    ext_vect_df = pd.DataFrame(columns=similarity_cols)
    lab_vect_df = pd.DataFrame(columns=similarity_cols)

    for col in similarity_cols:
        ext_vect_df[col] = ext_group[col].apply(vectorize, unique_values=UNIQUE_DICT[col])
        lab_vect_df[col] = lab_group[col].apply(vectorize, unique_values=UNIQUE_DICT[col])

    #initialize
    reid_match_ext_group= np.array([])
    reid_match_lab_group = np.array([])
    accuracy_matrix_group = []

    #split between not nans and nans for impactValue
    not_nan_ext_df, not_nan_lab_df, nan_ext_df, nan_lab_df = split_nans(ext_group, lab_group, "impactValue", nan_policy=nan_policy)
    if len(not_nan_ext_df) and len(not_nan_lab_df):
        ext_vect_df_notna = ext_vect_df.loc[not_nan_ext_df.index]
        lab_vect_df_notna = lab_vect_df.loc[not_nan_lab_df.index]

        reid_match_ext, reid_match_lab, accuracy_matrix = match_rows(not_nan_ext_df, not_nan_lab_df, ext_vect_df_notna, lab_vect_df_notna, matching_cols, similarity_cols,  matching_cols_weights_qt, geo_match=geo_match, value_match=value_match)

        #add aggregated similarity
        agg_sim_candidates = compute_weighted_sim(accuracy_matrix, matching_cols, matching_cols_weights_qt)
        accuracy_matrix = np.append(accuracy_matrix, agg_sim_candidates.reshape(-1,1), axis=1)##!! in which order columns are added

        #store results for group
        accuracy_matrix_group.append(accuracy_matrix)
        reid_match_ext_group = np.append(reid_match_ext_group, reid_match_ext)
        reid_match_lab_group = np.append(reid_match_lab_group, reid_match_lab)

    if len(nan_ext_df) and len(nan_lab_df):

        ext_vect_df_na = ext_vect_df.loc[nan_ext_df.index]
        lab_vect_df_na = lab_vect_df.loc[nan_lab_df.index]

        #turn off value match for nans
        reid_match_ext, reid_match_lab, accuracy_matrix = match_rows(nan_ext_df, nan_lab_df, ext_vect_df_na, lab_vect_df_na, matching_cols, similarity_cols, matching_cols_weights_ql,  geo_match=geo_match, value_match=None)

        #need to append cols of nan to keep dim consistent if there are no impactValues
        if value_match == "pre": #or value_match_post:
            accuracy_matrix = np.append(accuracy_matrix, np.full((accuracy_matrix.shape[0], 1), np.nan), axis=1)

        #add aggregated similarity
        agg_sim_candidates = compute_weighted_sim(accuracy_matrix, matching_cols, matching_cols_weights_ql)
        accuracy_matrix = np.append(accuracy_matrix, agg_sim_candidates.reshape(-1,1), axis=1)##!! in which order columns are added

        #store results for group
        accuracy_matrix_group.append(accuracy_matrix)
        reid_match_ext_group = np.append(reid_match_ext_group, reid_match_ext)
        reid_match_lab_group = np.append(reid_match_lab_group, reid_match_lab)

    #write as df
    id_accuracy_array = np.append(np.stack((reid_match_ext_group, reid_match_lab_group),axis=1), np.concatenate(accuracy_matrix_group), axis=1)
    match_idx.append(pd.DataFrame(id_accuracy_array,
                                  columns = ["ext_match_id", "lab_match_id"]+matching_cols+["match"]))

match_idx_df = pd.concat(match_idx)

#join extracted and labelled dataframes
matched_df = pd.concat([extracted_df.loc[match_idx_df["ext_match_id"].values].reset_index(drop=True),
                        labelled_df.loc[match_idx_df["lab_match_id"].values].add_suffix('_matched').reset_index(drop=True),
                        match_idx_df.reset_index(drop=True).add_suffix('_sim')], axis=1)
#overwrite / recompute impactValue sim and error
matched_df["impactValue_error"] = max_value_diff(matched_df["impactValue"].values, matched_df["impactValue_matched"].values)
#matched_df["impactValue_sim"] = calc_value_sim(matched_df["impactValue"].values, matched_df["impactValue_matched"].values)#[np.arange(len(matched_df)), np.arange(len(matched_df))]
if "geometry_sim" not in matched_df.columns:
    matched_df["geometry_sim"] = matched_df.apply(lambda x: IoU(x["geometry"], x["geometry_matched"]), axis=1)

if save_results:
    matched_df = delistify_cols(matched_df)
    matched_df.to_feather(DATA_OUT_PROC / (filename_out + ".feather"))


Processing appeal MDRBD022
Processing appeal MDRBJ019
Processing appeal MDRCM039
Processing appeal MDRCO023
Processing appeal MDRDZ011
Processing appeal MDRGE019
Processing appeal MDRGW003
Processing appeal MDRHU005
Processing appeal MDRIQ014
Processing appeal MDRKE058
Processing appeal MDRMY003
Processing appeal MDRMZ024
Processing appeal MDRNG041
Processing appeal MDRPH021
Processing appeal MDRPK026
Processing appeal MDRRW022
